# Clase 4 · 02 · Ingesta incremental y enriquecimiento

Ahora los eventos llegan como **archivos** JSON Lines a una carpeta del volumen, como si un sistema externo los depositara. Armamos un pipeline de dos consultas streaming:

```text
archivos stream_00X.jsonl ──Auto Loader──▶ stream_bronze_events (texto original)
                                               │
                                 parseo + calidad + join con clientes y productos
                                               ▼
                                   stream_silver_classified (válidos y cuarentena)
```

**Auto Loader** (`format("cloudFiles")`) recuerda en su checkpoint qué archivos ya leyó: cada ejecución procesa **sólo los archivos nuevos**. Es el equivalente a un offset, pero para archivos.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_rules

In [ ]:
%run ../common/streaming_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
for table in [customers, products]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá la clase 2."
T = lambda name: qualified_table(config, name)

In [ ]:
LAB = "ingesta"
reset_lab(spark, dbutils, config, LAB, ["stream_bronze_events", "stream_silver_classified"])
input_dir = lab_path(config, LAB, "input")
dbutils.fs.mkdirs(input_dir)
bronze_table, silver_table = T("stream_bronze_events"), T("stream_silver_classified")

## 1. Definir las dos consultas

- **Bronze** guarda cada línea tal como llegó, con el archivo de origen. Si mañana cambia el contrato, el dato original sigue disponible.
- **Silver** parsea el JSON, valida tipos y reglas, y hace un **join stream-tabla**: cada evento del stream se enriquece con el país del cliente (`silver_customers`) y la categoría del producto (`silver_products`), que son tablas estáticas de la clase 2.

In [ ]:
def run_bronze():
    files = (spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "text")
             .schema("value STRING")
             .load(input_dir))
    bronze = files.select(F.col("value").alias("raw_json"),
                          F.col("_metadata.file_path").alias("source_file"),
                          F.current_timestamp().alias("ingested_at"))
    return run_stream(bronze, lab_path(config, LAB, "checkpoints", "bronze"), table=bronze_table)

def run_silver():
    raw = spark.readStream.table(bronze_table)
    classified = classify_stream_events(raw, spark.table(customers), spark.table(products))   # join stream-tabla
    return run_stream(classified, lab_path(config, LAB, "checkpoints", "silver"), table=silver_table)

## 2. Cinco llegadas, una por vez

En cada vuelta el productor deposita un archivo nuevo y ejecutamos las dos consultas. Medimos cuántas filas nuevas procesó cada una.

In [ ]:
arrivals = []
for batch_id in ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"]:
    published = publish_arrival(dbutils, config, LAB, batch_id)
    bronze_new = input_rows(run_bronze())
    silver_new = input_rows(run_silver())
    arrivals.append({"llegada": batch_id, "filas en el archivo": published,
                     "nuevas en Bronze": bronze_new, "nuevas en Silver": silver_new})
    print(f"{batch_id}: archivo con {published} filas → Bronze +{bronze_new} → Silver +{silver_new}")
display(pd.DataFrame(arrivals))

Cada consulta procesó **sólo lo nuevo** de cada llegada, nunca los archivos anteriores. Si ejecutamos otra vez sin archivos nuevos, no hay nada pendiente:

In [ ]:
print("Reejecución sin archivos nuevos → Bronze +", input_rows(run_bronze()), "· Silver +", input_rows(run_silver()))

## 3. Calidad por llegada

In [ ]:
quality = (spark.table(silver_table)
           .groupBy("source_batch_id")
           .agg(F.sum(F.col("quality_reason").isNull().cast("int")).alias("válidos"),
                F.sum(F.col("quality_reason").isNotNull().cast("int")).alias("cuarentena"))
           .orderBy("source_batch_id").toPandas())

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(quality["source_batch_id"], quality["válidos"], color="#16a34a", label="válidos")
ax.bar(quality["source_batch_id"], quality["cuarentena"], bottom=quality["válidos"], color="#dc2626", label="cuarentena")
ax.set_ylabel("Eventos")
ax.set_title("Eventos clasificados por llegada")
ax.legend()
plt.show()

display(spark.table(silver_table).where("quality_reason IS NOT NULL")
        .groupBy("source_batch_id", "quality_reason").count().orderBy("source_batch_id"))

## 4. El enriquecimiento (join stream-tabla)

Los eventos llegaron sólo con `customer_id` y `product_id`. Gracias al join, Silver ya tiene `country` y `category`. Un `customer_id` que no existe en `silver_customers` queda en cuarentena como `UNKNOWN_CUSTOMER`.

In [ ]:
display(spark.table(silver_table)
        .where(F.col("event_id").endswith("_000"))
        .select("source_batch_id", "event_id", "event_ts", "customer_id", "country", "product_id", "category",
                "amount", "payment_channel", "quality_reason")
        .orderBy("source_batch_id", "event_ts"))

## Resumen

- El checkpoint de Auto Loader permite procesar **sólo archivos nuevos** y retomar después de una falla.
- Bronze conserva el dato crudo; Silver aplica el contrato y separa la **cuarentena**.
- El **join stream-tabla** enriquece cada evento con datos de referencia. Si la tabla de referencia cambia, los eventos siguientes ven la versión nueva; los ya procesados no se recalculan.
- Silver todavía tiene el duplicado `e002` y el tardío `late_bad`: los tratamos en los notebooks 03 y 04.